# Amazon ML Challenge 2026 — Final Safe Inference Notebook

**Goal:** replace the exact-only final inference with the validated blocking + two-stage similarity pipeline.

### Design constraints
- Uses the validated **~99% candidate-recall blocker** from the development work.
- Does **not** build a 10M-row Python index.
- TEST S2 and TEST S3 are streamed from disk.
- TEST S1 is indexed once.
- Cheap filtering happens before the expensive `calculate_similarity`.
- S2 and S3 are processed in **separate cells**.
- Every source chunk writes a checkpoint file, so an interrupted run can resume.
- RAM target: stay comfortably below the **20 GB** Kaggle limit.
- No cell intentionally loads all S2/S3 into RAM.

**Important:** run cells in order. The two full inference cells can take time, but they print progress every chunk and checkpoint their work.

## Cell 1 — Paths and imports

This cell only initializes paths and imports the provided solution modules.

No large dataset is loaded.

In [ ]:
import os
import sys
import gc
import csv
import json
import time
import pickle
import heapq
import re
from pathlib import Path
from collections import defaultdict

ROOT = Path(
    "/kaggle/input/datasets/tejasweerajput/"
    "amazon-ml-challenge-2026-student-resource/"
    "student_resource"
)

TRAIN_DIR = ROOT / "dataset" / "train"
TEST_DIR = ROOT / "dataset" / "test"
SOLUTION_DIR = ROOT / "amazon_ml_solution"
SRC_DIR = SOLUTION_DIR / "src"
UTILS_DIR = ROOT / "utils"

WORK_DIR = Path("/kaggle/working/amazon_ml_final")
OUTPUT_DIR = WORK_DIR / "output"
CHECKPOINT_DIR = WORK_DIR / "inference_checkpoints"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from normalize import normalize_business_name, normalize_address
from candidate_generator import make_name_keys, make_address_keys
from similarity import calculate_similarity

print("=" * 70)
print("ENVIRONMENT READY")
print("=" * 70)
print("ROOT:", ROOT)
print("TEST S1:", TEST_S1.exists(), TEST_S1)
print("TEST S2:", TEST_S2.exists(), TEST_S2)
print("TEST S3:", TEST_S3.exists(), TEST_S3)
print("RAM-safe checkpoint directory:", CHECKPOINT_DIR)
print("=" * 70)

## Cell 2 — Final configuration

These are the settings established during the earlier validation work.

- cheap threshold = **0.10**
- detailed threshold = **0.70**
- score = 0.50 name + 0.30 address + 0.15 address-number + 0.05 country

The blocker itself is **not** being changed here.

In [ ]:
CHEAP_THRESHOLD = 0.10
FINAL_THRESHOLD = 0.70

NAME_WEIGHT = 0.50
ADDRESS_WEIGHT = 0.30
NUMBER_WEIGHT = 0.15
COUNTRY_WEIGHT = 0.05

# Keep only this many highest-scoring candidates per S1 in the
# final candidate file. The matching decision still uses FINAL_THRESHOLD.
TOP_K_CANDIDATES = 20

# Streaming/checkpoint size.
SOURCE_CHUNK_ROWS = 250_000

CONFIG_PATH = WORK_DIR / "pipeline_config_final.json"

config = {
    "cheap_threshold": CHEAP_THRESHOLD,
    "final_threshold": FINAL_THRESHOLD,
    "top_k_candidates_per_s1": TOP_K_CANDIDATES,
    "source_chunk_rows": SOURCE_CHUNK_ROWS,
    "score_weights": {
        "name_similarity": NAME_WEIGHT,
        "address_similarity": ADDRESS_WEIGHT,
        "address_number_match": NUMBER_WEIGHT,
        "country_match": COUNTRY_WEIGHT,
    },
    "blocker": "validated name/address blocking; country included in key",
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))
print("Saved:", CONFIG_PATH)

## Cell 3 — Build compact TEST S1 index

This is the only large in-memory index.

It stores:
1. TEST S1 rows needed for scoring.
2. Name blocking key → S1 IDs.
3. Address blocking key → S1 IDs.
4. Exact normalized name/address indexes for the fast path.

S2/S3 are **not** loaded.

The key format exactly follows the existing TEST-S1 blocker:
`(country, key_type, key_value)`.

In [ ]:
print("=" * 70)
print("CELL 3 — BUILD TEST S1 INDEX")
print("=" * 70)

t0 = time.time()

# entity_id -> compact raw row
s1_rows = {}

# Pre-normalized S1 fields used by cheap filtering.
# This avoids normalizing the same S1 strings millions of times.
s1_norm = {}

# validated blocking indexes
s1_name_index = defaultdict(list)
s1_address_index = defaultdict(list)

# exact fast-path indexes
exact_name_index = defaultdict(list)
exact_address_index = defaultdict(list)

s1_count = 0
name_entries = 0
address_entries = 0

with open(TEST_S1, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        sid = row.get("entity_id", "").strip()
        if not sid:
            continue

        country = row.get("country", "").strip().upper()
        name = row.get("business_name", "")
        address = row.get("business_address", "")

        clean = {
            "entity_id": sid,
            "business_name": name,
            "business_address": address,
            "country": country,
        }
        s1_rows[sid] = clean

        # Pre-normalize once.
        nn = normalize_business_name(name)
        aa = normalize_address(address)

        compact = nn.replace(" ", "")
        s1_norm[sid] = {
            "name": nn,
            "address": aa,
            "compact": compact,
            "numbers": {
                x for x in aa.split() if x.isdigit()
            },
        }

        # Exact indexes
        if nn:
            exact_name_index[(country, nn)].append(sid)
        if aa:
            exact_address_index[(country, aa)].append(sid)

        # Validated blocker
        for key_type, key_value in make_name_keys(name):
            if key_value:
                s1_name_index[(country, key_type, key_value)].append(sid)
                name_entries += 1

        for key_type, key_value in make_address_keys(address):
            if key_value:
                s1_address_index[(country, key_type, key_value)].append(sid)
                address_entries += 1

        s1_count += 1

        if s1_count % 100_000 == 0:
            print(
                f"S1 indexed: {s1_count:,} | "
                f"elapsed={time.time()-t0:.1f}s"
            )

print("=" * 70)
print("S1 INDEX READY")
print("=" * 70)
print("S1 rows:", f"{s1_count:,}")
print("Name blocking keys:", f"{len(s1_name_index):,}")
print("Address blocking keys:", f"{len(s1_address_index):,}")
print("Name posting entries:", f"{name_entries:,}")
print("Address posting entries:", f"{address_entries:,}")
print("Exact name keys:", f"{len(exact_name_index):,}")
print("Exact address keys:", f"{len(exact_address_index):,}")
print("Runtime:", f"{time.time()-t0:.1f}s")
print("=" * 70)

gc.collect()

## Cell 4 — Scoring helpers

The detailed score calls the **same `calculate_similarity` function from the supplied solution**, rather than inventing a new similarity implementation.

The cheap filter is deliberately simple and avoids `SequenceMatcher`.

In [ ]:
def address_numbers(normalized_address):
    return {
        x for x in normalized_address.split()
        if x.isdigit()
    }

def cheap_score_from_normalized(
    s1_name, s1_addr, s2_name, s2_addr, key_count
):
    exact_name = bool(s1_name) and s1_name == s2_name
    exact_address = bool(s1_addr) and s1_addr == s2_addr

    nums1 = address_numbers(s1_addr)
    nums2 = address_numbers(s2_addr)
    number_match = bool(nums1 and nums2 and (nums1 & nums2))

    c1 = s1_name.replace(" ", "")
    c2 = s2_name.replace(" ", "")

    prefix6 = (
        len(c1) >= 6 and len(c2) >= 6 and c1[:6] == c2[:6]
    )
    prefix8 = (
        len(c1) >= 8 and len(c2) >= 8 and c1[:8] == c2[:8]
    )

    score = (
        1.00 * exact_name
        + 1.00 * exact_address
        + 0.70 * number_match
        + 0.50 * prefix8
        + 0.30 * prefix6
        + 0.05 * min(key_count, 10)
    )

    return score

def detailed_score(s1_row, s2_row):
    f = calculate_similarity(s1_row, s2_row)
    score = (
        NAME_WEIGHT * f["name_similarity"]
        + ADDRESS_WEIGHT * f["address_similarity"]
        + NUMBER_WEIGHT * f["address_number_match"]
        + COUNTRY_WEIGHT * f["country_match"]
    )
    return score, f

print("Scoring helpers ready.")

## Cell 5 — Small smoke test

This is intentionally tiny.

It does **not** scan the full dataset. It verifies:
- blocking keys work,
- exact lookup works,
- detailed scoring works,
- output structures are valid.

If this cell fails, **do not start the full inference cells**.

In [ ]:
print("=" * 70)
print("SMOKE TEST")
print("=" * 70)

# Take the first TEST S1 row
sample_s1 = next(iter(s1_rows.values()))

country = sample_s1["country"]
nn = normalize_business_name(sample_s1["business_name"])
aa = normalize_address(sample_s1["business_address"])

exact_ids = set(exact_name_index.get((country, nn), []))
exact_ids.update(exact_address_index.get((country, aa), []))

block_ids = set()

for kt, kv in make_name_keys(sample_s1["business_name"]):
    block_ids.update(s1_name_index.get((country, kt, kv), []))

for kt, kv in make_address_keys(sample_s1["business_address"]):
    block_ids.update(s1_address_index.get((country, kt, kv), []))

print("Sample S1:", sample_s1["entity_id"])
print("Exact candidates:", len(exact_ids))
print("Blocking candidates:", len(block_ids))

if exact_ids:
    sid = next(iter(exact_ids))
    score, feats = detailed_score(sample_s1, s1_rows[sid])
    print("Detailed self/exact score:", round(score, 4))
    print("Features:", feats)

print("✅ SMOKE TEST PASSED")

## Cell 6 — Process TEST S2 with checkpoints

For every TEST S2 row:

1. Try exact normalized name/address.
2. **Still run the validated blocker** so exact-name collisions do not hide fuzzy candidates.
3. Count blocking-key hits.
4. Apply cheap threshold **0.10**.
5. Apply the supplied detailed similarity scorer only to survivors.
6. Keep the best `TOP_K_CANDIDATES` candidates per S1 inside each checkpoint.
7. Save a checkpoint after every **250,000 S2 rows**.

### Runtime safety

A single notebook execution has a **50-minute guard**. If the guard is reached, the cell stops cleanly after the last completed checkpoint. Rerun the same cell and it will skip completed checkpoints.

TEST S2 is streamed; it is never loaded into RAM as a full DataFrame.

In [ ]:
def process_source(source_path, label, max_minutes=50):
    print("=" * 70)
    print(f"PROCESSING {label}")
    print(f"Time guard: {max_minutes} minutes per notebook run")
    print("=" * 70)

    source_path = Path(source_path)
    label_lower = label.lower()
    chunk_dir = CHECKPOINT_DIR / label_lower
    chunk_dir.mkdir(parents=True, exist_ok=True)

    expected_rows = {
        "s2": 4_887_273,
        "s3": 5_082_316,
    }.get(label_lower)

    expected_chunks = (
        math.ceil(expected_rows / SOURCE_CHUNK_ROWS)
        if expected_rows is not None
        else None
    )

    # Count existing checkpoints first.
    existing = sorted(chunk_dir.glob("chunk_*.pkl"))
    print("Existing checkpoints:", len(existing))

    if expected_chunks is not None and len(existing) >= expected_chunks:
        print(
            f"✅ {label_upper} already has all "
            f"{expected_chunks} checkpoints. Nothing to process."
        )
        return

    completed_chunks = len(existing)

    source_start = time.time()
    chunk_rows = []
    chunk_no = completed_chunks
    total_rows = completed_chunks * SOURCE_CHUNK_ROWS

    def score_chunk(rows, chunk_no):
        result = {}

        for s2row in rows:
            source_id = s2row["entity_id"]
            country = s2row["country"]
            raw_name = s2row["business_name"]
            raw_addr = s2row["business_address"]

            s2_name = normalize_business_name(raw_name)
            s2_addr = normalize_address(raw_addr)
            s2_compact = s2_name.replace(" ", "")
            s2_numbers = {
                x for x in s2_addr.split() if x.isdigit()
            }

            # candidate -> number of independent blocking hits
            key_hits = {}

            # ------------------------------------------------
            # Exact normalized fast path
            # ------------------------------------------------
            if s2_name:
                for sid in exact_name_index.get(
                    (country, s2_name), []
                ):
                    key_hits[sid] = 10

            if s2_addr:
                for sid in exact_address_index.get(
                    (country, s2_addr), []
                ):
                    key_hits[sid] = max(
                        key_hits.get(sid, 0), 10
                    )

            # ------------------------------------------------
            # ALWAYS add validated blocker hits.
            #
            # This is important: an exact-name collision must
            # not prevent a genuine fuzzy/blocking candidate
            # from being considered.
            # ------------------------------------------------
            for kt, kv in make_name_keys(raw_name):
                if not kv:
                    continue

                for sid in s1_name_index.get(
                    (country, kt, kv), []
                ):
                    key_hits[sid] = key_hits.get(sid, 0) + 1

            for kt, kv in make_address_keys(raw_addr):
                if not kv:
                    continue

                for sid in s1_address_index.get(
                    (country, kt, kv), []
                ):
                    key_hits[sid] = key_hits.get(sid, 0) + 1

            # ------------------------------------------------
            # Cheap filter + detailed score
            # ------------------------------------------------
            for sid, hit_count in key_hits.items():
                sn = s1_norm.get(sid)
                s1 = s1_rows.get(sid)

                if sn is None or s1 is None:
                    continue

                cheap = cheap_score_from_normalized(
                    sn["name"],
                    sn["address"],
                    s2_name,
                    s2_addr,
                    hit_count
                )

                if cheap < CHEAP_THRESHOLD:
                    continue

                # Detailed scorer is the supplied solution scorer.
                score, _ = detailed_score(
                    s1,
                    {
                        "entity_id": source_id,
                        "business_name": raw_name,
                        "business_address": raw_addr,
                        "country": country,
                    }
                )

                # Retain moderately strong candidates so the
                # final top-K set is not identical to the match set.
                if score >= 0.50:
                    result.setdefault(sid, []).append(
                        (float(score), source_id)
                    )

        # Top-K inside this checkpoint.
        for sid, vals in result.items():
            vals.sort(reverse=True)
            result[sid] = vals[:TOP_K_CANDIDATES]

        out = chunk_dir / f"chunk_{chunk_no:04d}.pkl"
        tmp = out.with_suffix(".tmp")

        with open(tmp, "wb") as f:
            pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

        os.replace(tmp, out)

        print(
            f"{label.upper()}: checkpoint {chunk_no:04d} saved | "
            f"S1s={len(result):,} | {out.name}"
        )

    # --------------------------------------------------------
    # Stream source. Completed checkpoints are skipped.
    # --------------------------------------------------------
    with open(
        source_path,
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as f:
        reader = csv.DictReader(f, delimiter="\t")

        # Resume after completed checkpoints instead of rereading
        # already processed source rows.
        rows_to_skip = completed_chunks * SOURCE_CHUNK_ROWS
        if rows_to_skip:
            print(
                f"Resuming {label_upper}: skipping "
                f"{rows_to_skip:,} completed rows..."
            )
            for _ in range(rows_to_skip):
                try:
                    next(reader)
                except StopIteration:
                    break

        for row in reader:
            clean = {
                "entity_id": row.get("entity_id", "").strip(),
                "business_name": row.get("business_name", ""),
                "business_address": row.get("business_address", ""),
                "country": row.get("country", "").strip().upper(),
            }

            if not clean["entity_id"]:
                continue

            chunk_rows.append(clean)
            total_rows += 1

            if len(chunk_rows) >= SOURCE_CHUNK_ROWS:
                chunk_no += 1
                out = chunk_dir / f"chunk_{chunk_no:04d}.pkl"

                if out.exists():
                    print(
                        f"{label.upper()}: skip checkpoint "
                        f"{chunk_no:04d}"
                    )
                else:
                    score_chunk(chunk_rows, chunk_no)

                chunk_rows.clear()
                gc.collect()

                elapsed = time.time() - source_start
                print(
                    f"{label.upper()}: {total_rows:,} source rows | "
                    f"{elapsed/60:.2f} min"
                )

                if elapsed >= max_minutes * 60:
                    print(
                        f"⏸️ Time guard reached at "
                        f"{elapsed/60:.2f} min."
                    )
                    print(
                        "Rerun this SAME cell to continue; "
                        "completed checkpoints will be skipped."
                    )
                    return

    if chunk_rows:
        chunk_no += 1
        out = chunk_dir / f"chunk_{chunk_no:04d}.pkl"

        if out.exists():
            print(
                f"{label.upper()}: skip checkpoint "
                f"{chunk_no:04d}"
            )
        else:
            score_chunk(chunk_rows, chunk_no)

        chunk_rows.clear()
        gc.collect()

    print("=" * 70)
    print(
        f"{label.upper()} COMPLETE | rows={total_rows:,} | "
        f"chunks={chunk_no} | "
        f"time={(time.time()-source_start)/60:.2f} min"
    )
    print("=" * 70)


process_source(TEST_S2, "S2")

## Cell 7 — Process TEST S3 with checkpoints

Same pipeline as S2, but S3 is processed separately.

A 50-minute time guard is used here too. If the guard fires, simply rerun the cell; completed checkpoints are skipped.

This keeps the peak RAM stable and prevents a failed S2/S3 combined run from losing both.

In [ ]:
process_source(TEST_S3, "S3")
gc.collect()
print("✅ S3 processing finished.")

## Cell 8 — Merge checkpoints and make the final two required TSVs

This cell does not reread S2/S3.

It merges the small checkpoint files, keeps the highest-scoring candidates per S1, and writes:

- `matching_results.tsv`
- `candidate_pairs.tsv`

Matching rule:
- candidate score >= **0.70**
- multiple S2/S3 matches are allowed
- candidates are always a superset of final matches

The candidate file contains the retained top-K scored candidates, rather than only the final matches.

In [ ]:
print("=" * 70)
print("MERGING CHECKPOINTS")
print("=" * 70)

# s1_id -> dict(source_id -> score)
merged = {}

for label in ["s2", "s3"]:
    chunk_dir = CHECKPOINT_DIR / label

    files = sorted(chunk_dir.glob("chunk_*.pkl"))

    print(f"{label.upper()} checkpoint files:", len(files))

    for path in files:
        with open(path, "rb") as f:
            part = pickle.load(f)

        for sid, vals in part.items():
            bucket = merged.setdefault(sid, {})

            for score, source_id in vals:
                old = bucket.get(source_id)

                if old is None or score > old:
                    bucket[source_id] = score

        del part
        gc.collect()

print("S1s with retained candidates:", f"{len(merged):,}")

# ------------------------------------------------------------
# Finalize top-K candidate lists
# ------------------------------------------------------------
final_candidates = {}

for sid, score_map in merged.items():
    vals = [
        (float(score), str(source_id))
        for source_id, score in score_map.items()
    ]

    vals.sort(reverse=True)
    final_candidates[sid] = vals[:TOP_K_CANDIDATES]

# ------------------------------------------------------------
# Write required files
# ------------------------------------------------------------
MATCHING_PATH = OUTPUT_DIR / "matching_results.tsv"
CANDIDATE_PATH = OUTPUT_DIR / "candidate_pairs.tsv"

match_rows = 0
nonempty_matches = 0
candidate_rows = 0

with open(
    MATCHING_PATH, "w", encoding="utf-8", newline=""
) as fm, open(
    CANDIDATE_PATH, "w", encoding="utf-8", newline=""
) as fc:

    fm.write("source1_entity_id\tmatched_entity_ids\n")
    fc.write("source1_entity_id\tcandidate_entity_ids\n")

    # Preserve exact TEST S1 order.
    for sid in s1_rows.keys():
        vals = final_candidates.get(sid, [])

        candidate_ids = [
            source_id
            for score, source_id in vals
        ]

        match_ids = [
            source_id
            for score, source_id in vals
            if score >= FINAL_THRESHOLD
        ]

        fc.write(
            sid + "\t" + ",".join(sorted(set(candidate_ids))) + "\n"
        )

        fm.write(
            sid + "\t" + ",".join(sorted(set(match_ids))) + "\n"
        )

        candidate_rows += len(candidate_ids)
        match_rows += len(match_ids)

        if match_ids:
            nonempty_matches += 1

print("=" * 70)
print("FINAL OUTPUTS WRITTEN")
print("=" * 70)
print("Matching:", MATCHING_PATH)
print("Candidates:", CANDIDATE_PATH)
print("S1 rows:", f"{len(s1_rows):,}")
print("Nonempty matching S1:", f"{nonempty_matches:,}")
print("Total matched IDs:", f"{match_rows:,}")
print("Total candidate IDs:", f"{candidate_rows:,}")
print("=" * 70)

gc.collect()

## Cell 9 — Official validator

Run this before packaging.

This is a lightweight structural check compared with inference.

In [ ]:
import subprocess

validator = UTILS_DIR / "validate_submission.py"

result = subprocess.run(
    [
        sys.executable,
        str(validator),
        "--matching", str(MATCHING_PATH),
        "--candidate", str(CANDIDATE_PATH),
        "--test-dir", str(TEST_DIR),
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Official validator failed with return code {result.returncode}"
    )

print("✅ OFFICIAL VALIDATION PASSED")

## Cell 10 — Package final submission

Only run this after Cell 9 passes.

In [ ]:
import shutil
import zipfile

PACKAGE = WORK_DIR / "submission_package"
PACKAGE_CODE = PACKAGE / "code" / "business_entity_resolution"
PACKAGE_SRC = PACKAGE_CODE / "src"

if PACKAGE.exists():
    shutil.rmtree(PACKAGE)

PACKAGE_SRC.mkdir(parents=True, exist_ok=True)
(PACKAGE / "output").mkdir(parents=True, exist_ok=True)

shutil.copy2(
    MATCHING_PATH,
    PACKAGE / "output" / "matching_results.tsv"
)
shutil.copy2(
    CANDIDATE_PATH,
    PACKAGE / "output" / "candidate_pairs.tsv"
)

# Copy supplied source code.
for p in SRC_DIR.glob("*.py"):
    shutil.copy2(p, PACKAGE_SRC / p.name)

with open(
    PACKAGE_CODE / "README.md",
    "w",
    encoding="utf-8"
) as f:
    f.write("""# Amazon ML Challenge 2026 — Business Entity Resolution

Pipeline:
1. TEST S1 blocking index
2. Exact normalized name/address fast path
3. Validated multi-key blocking
4. Cheap structural filtering
5. Detailed similarity scoring
6. Top-K candidate retention
7. Thresholded final matching

The output files are under `output/`.
""")

with open(
    PACKAGE_CODE / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("pandas\nnumpy\n")

with open(
    PACKAGE / "Documentation_template.md",
    "w",
    encoding="utf-8"
) as f:
    f.write("""# Amazon ML Challenge 2026

## Approach
Validated multi-key business-name/address blocking followed by a two-stage
entity similarity scorer.

## Matching
Final score threshold: 0.70.

## Candidate generation
Validated blocking keys plus an exact normalized name/address fast path.
""")

ZIP_PATH = WORK_DIR / "amazon_ml_final_submission.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for root, _, files in os.walk(PACKAGE):
        for filename in files:
            full = Path(root) / filename
            z.write(full, full.relative_to(PACKAGE))

print("=" * 70)
print("PACKAGE READY")
print("=" * 70)
print("ZIP:", ZIP_PATH)
print(
    "Size:",
    f"{ZIP_PATH.stat().st_size / 1024**2:.2f} MB"
)
print("=" * 70)

## Cell 11 — Final sanity report

No inference is performed here.

In [ ]:
print("=" * 70)
print("FINAL SANITY REPORT")
print("=" * 70)

for p in [MATCHING_PATH, CANDIDATE_PATH]:
    print(
        p.name,
        "| exists =", p.exists(),
        "| size MB =", round(p.stat().st_size / 1024**2, 2)
        if p.exists() else "-"
    )

print("ZIP:", ZIP_PATH.exists())
print("Checkpoint directories:")
for label in ["s2", "s3"]:
    d = CHECKPOINT_DIR / label
    print(label.upper(), "chunks =", len(list(d.glob("chunk_*.pkl"))) if d.exists() else 0)

print("=" * 70)
print("DONE — upload the ZIP shown above.")
print("=" * 70)